# 08 · Feature engineering — CICIoT2023 (`IoT23`)
**Owner:** Yassine Mtibaa · **Phase 5 — Feature engineering** · Cross-Network Intrusion Analysis (CRISP-DM)

CICIoT2023 is analysed separately from the merged NetFlow data (different row unit and features, see `docs/datasets.md`),
so the whole of phase 5 is done here for this dataset, task by task in the order of `docs/TASK_BOARD.md` (lab Parts 3–5):

| Task | Section | Fitted on |
|---|---|---|
| 5.1 Train/test split (stratified by `attack_label`, grouped by feature vector) | §1 | — |
| 5.2 Time features | §2 | nothing (row-wise formulas) |
| 5.3 Categorical features (one-hot protocol, rare categories grouped) | §3 | train |
| 5.4 Numeric features (size spread, binning; why no polynomial terms) | §4 | train (bins) |
| 5.5 Domain-specific features (TCP handshake, application share) | §5 | nothing |
| 5.6 Scaling / transformation (log / Yeo-Johnson + RobustScaler) | §6 | train |
| 5.7 Feature validation (before/after, new features vs target) | §7 | — |
| 5.8 Feature selection (identities, artefacts, near-constant, correlation, importance) | §8 | train |
| 5.9 Export final dataset + data dictionary + usage | §9 | — |
| 5.10 Bonus: PCA view of class separability | §10 | train |

**Golden rule (no data leakage):** the split comes first. Every value that is *learned from data* — protocol categories, bin edges,
transformation parameters, scaler medians and IQRs, the selected features — is learned on the **training set only** and then applied unchanged to the test set.

**Data:** `data/processed/IoT23_clean.parquet` from notebook 05 (cleaned, deduplicated, at most 200,000 rows per attack label).

In [ ]:
import sys; sys.path.append("..")
from src.config import *

import warnings; warnings.filterwarnings("ignore")
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, PowerTransformer, KBinsDiscretizer
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.decomposition import PCA
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 80)

SOURCE = "IoT23"
df = pd.read_parquet(PROCESSED_DIR / f"{SOURCE}_clean.parquet")
for c in ["attack_family", "attack_label"]:
    df[c] = df[c].astype(str)
df["Protocol_Type"] = df["Protocol_Type"].astype(int)

LABEL_COLS = ["Label", "source_dataset", "attack_label", "attack_family", "is_attack", "conflicting_features"]
RAW_FEATURES = [c for c in df.columns if c not in LABEL_COLS]                                # the 39 window features
FLAGS = ["fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number", "ack_flag_number", "ece_flag_number", "cwr_flag_number"]
APP = ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC"]
PROTO_SHARES = ["TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"]
COUNTS = ["ack_count", "syn_count", "fin_count", "rst_count"]
SHARES = FLAGS + APP + PROTO_SHARES                                                           # shares of packets in the window, already in [0, 1]
CONT_RAW = [c for c in RAW_FEATURES if c not in SHARES + ["Protocol_Type"]]                    # sizes, rates, times, counts
assert len(RAW_FEATURES) == 39 and len(CONT_RAW) + len(SHARES) + 1 == 39

FAMILY_ORDER = [f for f in [BENIGN, "Denial of service", "Malware and botnets", "Reconnaissance",
                            "Interception and spoofing", "Web and injection", "Credential attacks"] if f in set(df["attack_family"])]
PAL = dict(zip(FAMILY_ORDER, sns.color_palette("tab10", len(FAMILY_ORDER))))
PROTO_NAMES = {0: "nonIP", 1: "ICMP", 2: "IGMP", 6: "TCP", 17: "UDP", 47: "GRE"}

def bal_index(d, n, frac=0.7, exclude=None):
    "Index of a family-balanced random sample: at most n rows and at most frac of each family (keeps the original index)."
    if exclude is not None:
        d = d.drop(index=exclude)
    parts = [g.sample(min(n, int(frac * len(g)) if frac < 1 else len(g)), random_state=RANDOM_STATE).index
             for _, g in d.groupby("attack_family", observed=True)]
    return np.concatenate(parts)

def savefig(name):
    plt.savefig(FIG_DIR / f"IoT23_fe_{name}.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"{len(df):,} rows | {len(RAW_FEATURES)} raw features: {len(CONT_RAW)} continuous/count, {len(SHARES)} shares, 1 protocol code")

---
## 1 · Train/test split (5.1)
The split is the **first** step of feature engineering: anything computed before it could carry information from the test set into training.

**How:** 80 % train / 20 % test, **stratified by `attack_label`** so that each of the 34 labels keeps the same share in both sets
(important for the small labels such as brute force and web attacks).

**Grouped by feature vector.** Notebook 05 removed exact duplicate rows, but some feature vectors still appear more than once because they carry
*different labels* (the `conflicting_features` rows, mostly DoS vs DDoS). If one copy went to train and another to test, the test set would contain
a vector the model has already seen. So every distinct feature vector is a **group**, and a whole group goes to one side. This follows the team decision
for NetFlow ("split 80/20 grouped by behaviour", decision log 2026-10-24).

In [ ]:
df["group_id"] = pd.util.hash_pandas_object(df[RAW_FEATURES], index=False).to_numpy()
gsize = df["group_id"].value_counts()
print(f"rows: {len(df):,} | distinct feature vectors (groups): {len(gsize):,}")
print(f"groups with more than one row: {(gsize > 1).sum():,}, holding {gsize[gsize > 1].sum():,} rows "
      f"(rows flagged conflicting_features: {df['conflicting_features'].sum():,})")

# one stratification key per group: its first label in alphabetical order
key = df[["group_id", "attack_label"]].sort_values("attack_label", kind="stable").drop_duplicates("group_id")
g_train, g_test = train_test_split(key["group_id"].to_numpy(), test_size=0.20,
                                   stratify=key["attack_label"], random_state=RANDOM_STATE)
df["split"] = np.where(np.isin(df["group_id"].to_numpy(), g_test), "test", "train")
overlap = len(set(df.loc[df.split == "train", "group_id"]) & set(df.loc[df.split == "test", "group_id"]))
print(f"train: {(df.split == 'train').sum():,} rows | test: {(df.split == 'test').sum():,} rows | "
      f"feature vectors in both sets: {overlap}")

In [ ]:
lab = pd.crosstab(df["attack_label"], df["split"])
lab["test share %"] = (100 * lab["test"] / lab.sum(axis=1)).round(2)
print(f"test share per label: min {lab['test share %'].min()} %, max {lab['test share %'].max()} %")
fam = pd.crosstab(df["attack_family"], df["split"]).loc[FAMILY_ORDER]
fam["test share %"] = (100 * fam["test"] / fam.sum(axis=1)).round(2)
fam["train % of rows"] = (100 * fam["train"] / fam["train"].sum()).round(2)
fam["test % of rows"] = (100 * fam["test"] / fam["test"].sum()).round(2)
fam.to_csv(TABLE_DIR / "IoT23_fe_split.csv")
conf = pd.crosstab(df["conflicting_features"], df["split"])
display(fam, conf)
train = df[df.split == "train"].drop(columns="split").reset_index(drop=True)
test = df[df.split == "test"].drop(columns="split").reset_index(drop=True)
del df, key, gsize

**Interpretation.** _(to be written: split)_

---
## 2 · Time features (5.2)
The task asks for duration, packets per second and bytes per second, and **no hour-of-day** feature.
CICIoT2023 has **no timestamps** at all (notebook 05, 2.1), so there is nothing to convert to a date and hour-of-day is not possible —
which is also what we want, because it would only learn the lab's attack schedule.

What the dataset does have: `IAT` (mean time between packets in the window, seconds) and `Rate` (packets per second).
First we check how the two are related, then we build the two time features that are missing.

In [ ]:
chk = train.sample(300_000, random_state=RANDOM_STATE)
prod = chk["Rate"].astype("float64") * chk["IAT"].astype("float64")
rho = stats.spearmanr(chk["Rate"], chk["IAT"]).statistic
print(f"Rate x IAT: median {prod.median():.4f}, P5 {prod.quantile(.05):.4f}, P95 {prod.quantile(.95):.4f}, "
      f"share within 5 % of 1: {100 * (prod.sub(1).abs() < .05).mean():.1f} % | Spearman rho(Rate, IAT) = {rho:.4f}")

**New time features** (row-wise formulas, nothing is learned from the data, so they can be computed on train and test alike):

| Feature | Formula | Meaning |
|---|---|---|
| `byte_rate` | `AVG` × `Rate` | bytes per second (mean packet size × packets per second) — throughput of the window |
| `window_duration` | `Number` × `IAT` | time spanned by the window, in seconds |

`window_duration` contains `Number` (10 packets for non-flood captures, 100 for floods — a capture artefact found in notebook 07, §6),
so we expect it to inherit the artefact; §7 and §8 check this.

In [ ]:
def add_time_features(d):
    d["byte_rate"] = (d["AVG"].astype("float64") * d["Rate"].astype("float64")).astype("float32")
    d["window_duration"] = (d["Number"].astype("float64") * d["IAT"].astype("float64")).astype("float32")
    return d

train, test = add_time_features(train), add_time_features(test)
TIME_FEATS = ["byte_rate", "window_duration"]
tf = train.groupby("attack_family")[["Rate", "IAT", "byte_rate", "window_duration", "Number"]].median().loc[FAMILY_ORDER]
tf.to_csv(TABLE_DIR / "IoT23_fe_time_features.csv")
tf.style.format("{:,.4g}")

**Interpretation.** _(to be written: time)_

---
## 3 · Categorical features (5.3)
CICIoT2023 has a single categorical column, `Protocol_Type` (the IP protocol number of the window, stored as a category in notebook 05).
It is a **code, not a quantity** (17 = UDP is not "more" than 6 = TCP), so it is **one-hot encoded**.

* The encoder is **fitted on train**; categories seen fewer than 1,000 times in train are grouped into `proto_other` (rare-category grouping),
  and any protocol that appears only in test also goes there (`handle_unknown="infrequent_if_exist"`).
* Frequency / target encoding of ports and application protocols (planned for NetFlow) does **not apply**: CICIoT2023 has no port column,
  and the application protocols (`HTTP`, `DNS`, …) are already numeric shares of packets.

In [ ]:
print("train counts per protocol:", train["Protocol_Type"].map(PROTO_NAMES).value_counts().to_dict())
ohe = OneHotEncoder(min_frequency=1000, handle_unknown="infrequent_if_exist", sparse_output=False)
ohe.fit(train[["Protocol_Type"]])
OHE_NAMES = [n.replace("Protocol_Type_infrequent_sklearn", "proto_other") for n in ohe.get_feature_names_out()]
OHE_NAMES = [f"proto_{PROTO_NAMES[int(n.split('_')[-1])]}" if n != "proto_other" else n for n in OHE_NAMES]
print("one-hot columns:", OHE_NAMES, "| grouped as rare:", [PROTO_NAMES[int(v)] for v in ohe.infrequent_categories_[0]])
oh_check = pd.DataFrame(ohe.transform(test[["Protocol_Type"]].head(5)), columns=OHE_NAMES)
oh_check.insert(0, "Protocol_Type", test["Protocol_Type"].head(5).map(PROTO_NAMES).values)
oh_check

**Interpretation.** _(to be written: categorical)_

---
## 4 · Numeric features (5.4)
**Ratios / spread of packet sizes.** The window statistics give the mean, minimum, maximum and standard deviation of packet sizes.
Two combinations describe *how uniform* the packets are, independently of their absolute size:

| Feature | Formula | Meaning |
|---|---|---|
| `size_range` | `Max` − `Min` | gap between the largest and smallest packet (bytes) |
| `size_cv` | `Std` / `AVG` | coefficient of variation of packet size (no unit): 0 = all packets identical |

(`AVG` is at least 46 bytes, so the division is always defined.)

**Binning.** `Rate` is cut into 5 quantile bins **learned on train** (`rate_level` 0–4). The bins make the rate easy to report
("most floods are in the top bin") but they carry less information than `Rate` itself, so `rate_level` is kept for reporting only, not as a model feature.

**Polynomial terms: not created.** They would multiply the number of columns (≈ 900 for 40 features at degree 2), many inputs are shares in [0, 1] or exact
copies of each other (§8), and the flood patterns are thresholds on single features (notebook 07, §6), which tree-based models capture without squared terms.

In [ ]:
def add_numeric_features(d):
    d["size_range"] = (d["Max"] - d["Min"]).astype("float32")
    d["size_cv"] = (d["Std"].astype("float64") / d["AVG"].astype("float64")).astype("float32")
    return d

train, test = add_numeric_features(train), add_numeric_features(test)
NUM_FEATS = ["size_range", "size_cv"]

kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf")
kb.fit(train[["Rate"]].astype("float64"))
for d in (train, test):
    d["rate_level"] = kb.transform(d[["Rate"]].astype("float64")).ravel().astype("int8")
print("Rate bin edges learned on train (packets/s):", np.round(kb.bin_edges_[0], 1))
bins = (100 * pd.crosstab(train["attack_family"], train["rate_level"], normalize="index")).loc[FAMILY_ORDER].round(1)
bins.columns = [f"level {c}" for c in bins.columns]
bins.to_csv(TABLE_DIR / "IoT23_fe_rate_bins.csv")
display(bins)
train.groupby("attack_family")[NUM_FEATS + ["AVG", "Std"]].median().loc[FAMILY_ORDER].round(3)

**Interpretation.** _(to be written: numeric)_

---
## 5 · Domain-specific features (5.5)
Built from how the attacks work (notebook 07, §6 fingerprints):

| Feature | Formula | Why |
|---|---|---|
| `syn_minus_ack` | `syn_flag_number` − `ack_flag_number` | share of half-open handshakes: SYN floods send SYNs that are never acknowledged (→ close to +1), normal TCP traffic is mostly ACKs (→ negative) |
| `app_share` | sum of the 7 application-protocol shares | share of packets that belong to a known application (HTTP, HTTPS, DNS, Telnet, SMTP, SSH, IRC): close to 0 for raw floods, high for web, brute-force and normal traffic |
| `control_share` | `syn` + `fin` + `rst` flag shares | connection set-up / tear-down packets per data packet: scans and floods are dominated by control packets |

**Not possible with this dataset** (planned for NetFlow): *traffic asymmetry* needs the direction of each packet (in/out bytes), and *burstiness*
needs the spread of inter-arrival times; CICIoT2023 only gives the mean `IAT` of the window.

In [ ]:
def add_domain_features(d):
    d["syn_minus_ack"] = (d["syn_flag_number"] - d["ack_flag_number"]).astype("float32")
    d["app_share"] = d[APP].sum(axis=1).astype("float32")
    d["control_share"] = d[["syn_flag_number", "fin_flag_number", "rst_flag_number"]].sum(axis=1).astype("float32")
    return d

train, test = add_domain_features(train), add_domain_features(test)
DOMAIN_FEATS = ["syn_minus_ack", "app_share", "control_share"]
dom = train.groupby("attack_family")[DOMAIN_FEATS].median().loc[FAMILY_ORDER]
dom.to_csv(TABLE_DIR / "IoT23_fe_domain_features.csv")
dom.round(3)

**Interpretation.** _(to be written: domain)_

---
## 6 · Scaling and transformation (5.6)
**Which columns.** Only the continuous and count columns (sizes, rates, times, counts and the new time / size features) are transformed and scaled.
The 22 shares, `syn_minus_ack`, `app_share`, `control_share` and the one-hot protocol columns are already on a small fixed range ([0, 1], [−1, 1] or [0, 3])
and keep their meaning ("60 % of the packets are SYN"), so they are left as they are.

**Transformation (shape).** Notebook 07 showed extreme right skew (skewness up to ≈ 1,800). Rule, decided on a 500,000-row sample of **train**:
* |skewness| ≤ 1 → no transformation;
* otherwise `log1p(x)` = log(1 + x) (simple, interpretable, defined at 0) if it brings |skewness| to ≤ 1;
* if it does not, the better of `log1p` and **Yeo-Johnson** (a power transformation whose parameter λ is estimated on train).

**Scaling (range).** `RobustScaler`: subtract the **median** and divide by the **IQR**, both learned on train. We use it instead of
`StandardScaler` (mean and standard deviation) because in this data the outliers *are* the attacks (notebook 05, 2.4): with the mean and standard deviation the
flood windows would set the scale for everything else.

In [ ]:
CONT = CONT_RAW + TIME_FEATS + NUM_FEATS
BOUNDED = SHARES + DOMAIN_FEATS
assert (train[CONT] >= 0).all().all(), "log1p needs non-negative values"
fit_s = train.sample(500_000, random_state=RANDOM_STATE)

rows, TRANSFORM, PT = [], {}, {}
for c in CONT:
    x = fit_s[c].to_numpy("float64")
    s_raw, s_log = stats.skew(x), stats.skew(np.log1p(x))
    pt = PowerTransformer(method="yeo-johnson", standardize=False).fit(x.reshape(-1, 1))
    xy = pt.transform(x.reshape(-1, 1)).ravel()
    s_yj = stats.skew(xy) if np.isfinite(xy).all() else np.inf
    if abs(s_raw) <= 1:
        t = "none"
    elif abs(s_log) <= 1 or abs(s_log) <= abs(s_yj):
        t = "log1p"
    else:
        t, PT[c] = "yeo-johnson", pt
    TRANSFORM[c] = t
    rows.append({"feature": c, "skew raw": s_raw, "skew log1p": s_log, "skew Yeo-Johnson": s_yj, "chosen": t,
                 "lambda (YJ)": pt.lambdas_[0] if t == "yeo-johnson" else np.nan})
skew_tab = pd.DataFrame(rows).set_index("feature")

def apply_transforms(d):
    out = {}
    for c in CONT:
        x = d[c].to_numpy("float64")
        if TRANSFORM[c] == "log1p":
            x = np.log1p(x)
        elif TRANSFORM[c] == "yeo-johnson":
            x = PT[c].transform(x.reshape(-1, 1)).ravel()
        out[c] = x
    return pd.DataFrame(out, index=d.index)

scaler = RobustScaler().fit(apply_transforms(train))                      # medians and IQRs from TRAIN only
def build_X(d):
    Xc = pd.DataFrame(scaler.transform(apply_transforms(d)), columns=CONT, index=d.index)
    Xo = pd.DataFrame(ohe.transform(d[["Protocol_Type"]]), columns=OHE_NAMES, index=d.index)
    return pd.concat([Xc, d[BOUNDED], Xo], axis=1).astype("float32")

X_train, X_test = build_X(train), build_X(test)
skew_tab["skew after (train)"] = [stats.skew(X_train.loc[fit_s.index, c]) for c in CONT]
skew_tab.to_csv(TABLE_DIR / "IoT23_fe_transform.csv")
print("chosen transformation:", pd.Series(TRANSFORM).value_counts().to_dict(), "| matrix:", X_train.shape, X_test.shape)
skew_tab.round(2)

In [ ]:
q = lambda X: pd.DataFrame({"median": X[CONT].median(), "IQR": X[CONT].quantile(.75) - X[CONT].quantile(.25)})
chk_scale = pd.concat({"train": q(X_train), "test": q(X_test)}, axis=1).round(3)
chk_scale.to_csv(TABLE_DIR / "IoT23_fe_scaling_check.csv")
print("train medians all 0:", bool((chk_scale[("train", "median")].abs() < 1e-6).all()),
      "| train IQRs 1 or 0 (constant over the middle 50 %):", sorted(chk_scale[("train", "IQR")].round(3).unique()))
chk_scale

In [ ]:
SHOW = ["Rate", "IAT", "AVG", "Std", "Header_Length", "byte_rate"]
fig, axes = plt.subplots(2, len(SHOW), figsize=(22, 7))
for j, c in enumerate(SHOW):
    axes[0, j].hist(fit_s[c].astype("float64"), bins=60, color="tab:gray"); axes[0, j].set_yscale("log")
    axes[0, j].set_title(f"{c} — raw\nskew {skew_tab.loc[c, 'skew raw']:.1f}")
    axes[1, j].hist(X_train.loc[fit_s.index, c], bins=60, color="tab:blue")
    axes[1, j].set_title(f"{TRANSFORM[c]} + RobustScaler\nskew {skew_tab.loc[c, 'skew after (train)']:.2f}")
axes[0, 0].set_ylabel("rows (log scale)"); axes[1, 0].set_ylabel("rows")
plt.suptitle("Before / after transformation and scaling (500,000 training rows)", y=1.02)
plt.tight_layout(); savefig("before_after")

**Interpretation.** _(to be written: transform)_

---
## 7 · Feature validation (5.7)
Do the new features behave as intended, and do they say something about the target that the original columns did not?
All checks use the **training set only**, on a family-balanced sample (at most 15,000 rows per family, conflicting rows excluded) so that
Denial of service (≈ 60 % of rows) does not dominate.

In [ ]:
NEW_FEATS = TIME_FEATS + NUM_FEATS + DOMAIN_FEATS
pool = train[~train["conflicting_features"]]
BAL = bal_index(pool, 15_000)
VAL = bal_index(pool, 5_000, frac=1.0, exclude=BAL)                   # disjoint validation rows, still from train
yb, yv = train.loc[BAL, "attack_family"], train.loc[VAL, "attack_family"]
print(f"balanced sample: {len(BAL):,} rows | validation sample: {len(VAL):,} rows")
print(yb.value_counts().loc[FAMILY_ORDER].to_dict())

fig, axes = plt.subplots(1, len(NEW_FEATS), figsize=(26, 5), sharey=True)
for ax, c in zip(axes, NEW_FEATS):
    sns.boxplot(x=X_train.loc[BAL, c], y=yb, order=FAMILY_ORDER, palette=PAL, ax=ax, showfliers=False)
    ax.set_title(c); ax.set_xlabel("transformed / scaled" if c in CONT else "raw share"); ax.set_ylabel("")
plt.suptitle("New features by attack family (training set, balanced sample)", y=1.03)
plt.tight_layout(); savefig("new_features_by_family")

In [ ]:
PARENTS = {"byte_rate": ["Rate", "AVG"], "window_duration": ["IAT", "Number"], "size_range": ["Max", "Min"],
           "size_cv": ["Std", "AVG"], "syn_minus_ack": ["syn_flag_number", "ack_flag_number"],
           "app_share": ["HTTP", "HTTPS", "DNS"], "control_share": ["syn_flag_number", "rst_flag_number"]}
cols = list(dict.fromkeys(NEW_FEATS + [p for v in PARENTS.values() for p in v]))
mi_fam = pd.Series(mutual_info_classif(X_train.loc[BAL, cols], yb, random_state=RANDOM_STATE, n_jobs=-1), index=cols)
mi_att = pd.Series(mutual_info_classif(X_train.loc[BAL, cols], train.loc[BAL, "is_attack"], random_state=RANDOM_STATE, n_jobs=-1), index=cols)

def rank_biserial(x, y):
    "r > 0: attack windows tend to have larger values than benign ones (Mann-Whitney U based)."
    a, b = x[y == 1], x[y == 0]
    u = stats.mannwhitneyu(a, b, alternative="two-sided").statistic
    return 2 * u / (len(a) * len(b)) - 1

isat = train.loc[BAL, "is_attack"].to_numpy()
val = pd.DataFrame({"parents": [", ".join(PARENTS[c]) for c in NEW_FEATS],
                    "MI family (new)": mi_fam[NEW_FEATS].values,
                    "best MI family (parent)": [mi_fam[PARENTS[c]].max() for c in NEW_FEATS],
                    "MI is_attack (new)": mi_att[NEW_FEATS].values,
                    "rank-biserial r (attack vs benign)": [rank_biserial(X_train.loc[BAL, c].to_numpy(), isat) for c in NEW_FEATS]},
                   index=NEW_FEATS)
val["gain over parent"] = val["MI family (new)"] - val["best MI family (parent)"]
val.to_csv(TABLE_DIR / "IoT23_fe_validation.csv")
val.round(3)

**Interpretation.** _(to be written: validation)_

---
## 8 · Feature selection (5.8)
Selection is done on the **training set** (balanced sample, conflicting rows excluded), in five justified steps:

1. **Exact identities** — columns that are a formula of other columns carry no new information.
2. **Artefacts** — columns that describe the *capture method* or the *lab machines*, not the attack behaviour (notebook 07, §6). A model would use them as shortcuts that do not exist on another network.
3. **Near-constant columns** — one value in ≥ 99.5 % of rows, *unless* the rare non-zero values are concentrated in one family (then they are a rare-class signal, e.g. Telnet for brute force).
4. **Highly correlated pairs** (|Spearman ρ| ≥ 0.95) — keep the column with the higher mutual information with the family.
5. **Importance ranking** of what remains (mutual information + random-forest permutation importance) to justify the final set.

In [ ]:
s = train.loc[BAL]
ident = pd.DataFrame([
    ("Tot_size", "AVG", (s.Tot_size - s.AVG).abs().max()),
    ("Variance", "Std²", ((s.Variance - s.Std.astype("float64") ** 2).abs() / s.Variance.clip(lower=1)).max()),
    ("Tot_sum", "AVG × Number", ((s.Tot_sum - s.AVG.astype("float64") * s.Number).abs() / s.Tot_sum).max()),
    ("LLC", "IPv", (s.LLC - s.IPv).abs().max()),
    ("ARP", "1 − IPv", (s.ARP - (1 - s.IPv)).abs().max()),
] + [(c, f"{c.split('_')[0]}_flag_number × Number", (s[c] - (s[f"{c.split('_')[0]}_flag_number"] * s.Number).round()).abs().max())
     for c in COUNTS], columns=["column", "equals", "max difference"])
ident["identity holds"] = ident["max difference"] < 0.01
ident

In [ ]:
DROP_IDENTITY = {"Tot_size": "= AVG", "Variance": "= Std²", "LLC": "= IPv", "IPv": "= 1 − ARP (ARP kept: names the non-IP share directly)"}
CHECKED_BY = {"Tot_size": "Tot_size", "Variance": "Variance", "LLC": "LLC", "IPv": "ARP"}
DROP_IDENTITY = {k: v for k, v in DROP_IDENTITY.items() if ident.set_index("column").loc[CHECKED_BY[k], "identity holds"]}
DROP_ARTEFACT = {"Number": "packets per window: 10 for non-flood captures, 100 for floods (capture method)",
                 "Tot_sum": "= AVG × Number, inherits the Number artefact",
                 "window_duration": "= Number × IAT, inherits the Number artefact",
                 "Time_To_Live": "≈ 64 for every attacker, ≈ 99 for benign: identifies the lab machines' operating systems",
                 **{c: "= flag share × Number, inherits the Number artefact" for c in COUNTS}}

# shortcut demonstration: how far do the artefact columns alone get?
art_cols = ["Number", "Time_To_Live"]
rf_art = RandomForestClassifier(n_estimators=100, n_jobs=-1, class_weight="balanced", random_state=RANDOM_STATE)
rf_art.fit(X_train.loc[BAL, art_cols], yb)
print("balanced accuracy with Number + Time_To_Live only:",
      round(balanced_accuracy_score(yv, rf_art.predict(X_train.loc[VAL, art_cols])), 3), "(chance level: 1/7 = 0.143)")

In [ ]:
cand = [c for c in X_train.columns if c not in DROP_IDENTITY and c not in DROP_ARTEFACT]
top = X_train.loc[BAL, cand].apply(lambda x: x.value_counts(normalize=True).iloc[0])
near = top[top >= 0.995].index
nz_overall = (X_train.loc[BAL, near] != X_train.loc[BAL, near].mode().iloc[0]).mean()
nz_family = (X_train.loc[BAL, near] != X_train.loc[BAL, near].mode().iloc[0]).groupby(yb).mean().T[FAMILY_ORDER]
near_tab = pd.concat([top[near].rename("top value share"), nz_overall.rename("non-top share"), nz_family], axis=1)
near_tab["max lift"] = nz_family.max(axis=1) / nz_overall
near_tab["family"] = nz_family.idxmax(axis=1)
near_tab["decision"] = np.where(near_tab["max lift"] >= 3, "keep (rare-class signal)", "drop")
DROP_NEAR = {c: f"top value in {100 * near_tab.loc[c, 'top value share']:.2f} % of rows, no family concentration"
             for c in near_tab.index[near_tab["decision"] == "drop"]}
near_tab.round(3)

In [ ]:
cand2 = [c for c in cand if c not in DROP_NEAR]
mi_all = pd.Series(mutual_info_classif(X_train.loc[BAL, cand2], yb, random_state=RANDOM_STATE, n_jobs=-1), index=cand2)
rho = X_train.loc[BAL, cand2].corr(method="spearman")
pairs = (rho.where(np.triu(np.ones(rho.shape, bool), 1)).stack().rename("rho").reset_index())
pairs = pairs[pairs["rho"].abs() >= 0.95].assign(abs=lambda p: p["rho"].abs()).sort_values("abs", ascending=False)
DROP_CORR, kept = {}, []
for _, p in pairs.iterrows():
    a, b = p["level_0"], p["level_1"]
    if a in DROP_CORR or b in DROP_CORR:
        continue
    lose, keep = (a, b) if mi_all[a] < mi_all[b] else (b, a)
    DROP_CORR[lose] = f"|rho| = {abs(p['rho']):.3f} with {keep}, lower MI ({mi_all[lose]:.3f} vs {mi_all[keep]:.3f})"
corr_tab = pairs.rename(columns={"level_0": "feature A", "level_1": "feature B"})[["feature A", "feature B", "rho"]]
corr_tab["dropped"] = [a if a in DROP_CORR else b if b in DROP_CORR else "—" for a, b in zip(corr_tab["feature A"], corr_tab["feature B"])]
corr_tab.to_csv(TABLE_DIR / "IoT23_fe_correlated_pairs.csv", index=False)
corr_tab.round(4)

In [ ]:
FINAL = [c for c in cand2 if c not in DROP_CORR]
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, class_weight="balanced", random_state=RANDOM_STATE)
rf.fit(X_train.loc[BAL, FINAL], yb)
pv = rf.predict(X_train.loc[VAL, FINAL])
print(f"sanity check on validation rows from TRAIN (not a model evaluation): balanced accuracy "
      f"{balanced_accuracy_score(yv, pv):.3f}, macro F1 {f1_score(yv, pv, average='macro'):.3f}")
perm = permutation_importance(rf, X_train.loc[VAL, FINAL], yv, n_repeats=5, scoring="f1_macro",
                              random_state=RANDOM_STATE, n_jobs=-1)
rank = pd.DataFrame({"MI family": mi_all[FINAL], "RF impurity importance": rf.feature_importances_,
                     "permutation importance (macro F1 drop)": perm.importances_mean,
                     "perm. std": perm.importances_std}, index=FINAL).sort_values("permutation importance (macro F1 drop)", ascending=False)
rank["rank"] = range(1, len(rank) + 1)
rank.to_csv(TABLE_DIR / "IoT23_fe_importance.csv")
print(f"final feature set: {len(FINAL)} features")
rank.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 8.5), sharey=True)
r = rank.iloc[::-1]
ax[0].barh(r.index, r["permutation importance (macro F1 drop)"], xerr=r["perm. std"], color="tab:blue")
ax[0].set_title("Permutation importance (drop in macro F1)")
ax[1].barh(r.index, r["MI family"], color="tab:orange"); ax[1].set_title("Mutual information with attack family")
plt.suptitle(f"Ranking of the {len(FINAL)} selected features (training set, balanced sample)", y=1.01)
plt.tight_layout(); savefig("importance")

summary = pd.DataFrame([(c, "identity", r) for c, r in DROP_IDENTITY.items()] + [(c, "artefact", r) for c, r in DROP_ARTEFACT.items()] +
                       [(c, "near-constant", r) for c, r in DROP_NEAR.items()] + [(c, "correlated", r) for c, r in DROP_CORR.items()],
                       columns=["feature", "step", "reason"])
summary.to_csv(TABLE_DIR / "IoT23_fe_dropped.csv", index=False)
print(f"{X_train.shape[1]} candidate columns -> {len(FINAL)} selected ({len(summary)} dropped)")
summary

**Interpretation.** _(to be written: selection)_

---
## 9 · Export the final dataset, data dictionary and usage (5.9)

In [ ]:
KEEP_LABELS = ["attack_label", "attack_family", "is_attack", "conflicting_features"]
out_train = pd.concat([X_train[FINAL], train[KEEP_LABELS]], axis=1)
out_test = pd.concat([X_test[FINAL], test[KEEP_LABELS]], axis=1)
for name, d in [("train", out_train), ("test", out_test)]:
    d.to_parquet(PROCESSED_DIR / f"{SOURCE}_{name}_features.parquet", index=False)
joblib.dump({"transform": TRANSFORM, "yeo_johnson": PT, "scaler": scaler, "scaler_columns": CONT, "onehot": ohe,
             "onehot_names": OHE_NAMES, "rate_bins": kb, "final_features": FINAL, "random_state": RANDOM_STATE},
            PROCESSED_DIR / f"{SOURCE}_preprocessor.joblib")
print("saved:", [p.name for p in sorted(PROCESSED_DIR.glob(f"{SOURCE}_*features.parquet"))], f"{SOURCE}_preprocessor.joblib")
print(f"train {out_train.shape} | test {out_test.shape} | NaN: {int(out_train.isna().sum().sum() + out_test.isna().sum().sum())}")

In [ ]:
FORMULA = {"byte_rate": "AVG × Rate", "window_duration": "Number × IAT", "size_range": "Max − Min", "size_cv": "Std / AVG",
           "syn_minus_ack": "syn_flag_number − ack_flag_number", "app_share": "HTTP + HTTPS + DNS + Telnet + SMTP + SSH + IRC",
           "control_share": "syn + fin + rst flag shares", **{n: f"Protocol_Type == {n.split('_')[1]}" for n in OHE_NAMES}}
dropped = summary.set_index("feature")
dict_rows = []
for c in list(X_train.columns) + ["rate_level"]:
    dict_rows.append({
        "feature": c,
        "origin": "original" if c in RAW_FEATURES else "one-hot (5.3)" if c in OHE_NAMES else "derived",
        "formula": FORMULA.get(c, "binned Rate, 5 quantile bins (reporting only)" if c == "rate_level" else ""),
        "transformation": TRANSFORM.get(c, "none"),
        "scaling": "RobustScaler (train median / IQR)" if c in CONT else "none",
        "in final set": c in FINAL,
        "reason if not": dropped.loc[c, "reason"] if c in dropped.index else ("reporting only" if c == "rate_level" else ""),
    })
dictionary = pd.DataFrame(dict_rows)
dictionary.to_csv(TABLE_DIR / "IoT23_fe_dictionary.csv", index=False)
dictionary

**How to use the exported files**

```python
import pandas as pd
train = pd.read_parquet("data/processed/IoT23_train_features.parquet")
test  = pd.read_parquet("data/processed/IoT23_test_features.parquet")
train = train[~train["conflicting_features"]]          # decision log 2026-10-18: drop vectors that carry two labels before training
X_cols = [c for c in train.columns if c not in ["attack_label", "attack_family", "is_attack", "conflicting_features"]]
X, y = train[X_cols], train["attack_family"]           # or attack_label / is_attack
```
* The features are **already transformed and scaled** with parameters learned on train; do not refit anything on the test set.
* New raw data: `prep = joblib.load("data/processed/IoT23_preprocessor.joblib")`, add the derived features with the formulas of the dictionary,
  then apply `prep["transform"]` / `prep["yeo_johnson"]`, `prep["scaler"]` (columns `prep["scaler_columns"]`) and `prep["onehot"]`, and keep `prep["final_features"]`.
* The test set keeps the sample's class mix (≈ 95 % attacks). A *realistic* test set (e.g. 95 % benign / 5 % attacks, as decided for NetFlow) is drawn from it at modelling time.
* Full dictionary: `tables/IoT23_fe_dictionary.csv` and `docs/iot23_feature_set.md`.

---
## 10 · Bonus: PCA — a 2-D view of class separability (5.10)
PCA is fitted on a family-balanced **training** sample of the final features (already scaled, so no feature dominates by its unit).

In [ ]:
P_idx = bal_index(pool, 5_000)
pca = PCA(n_components=10, random_state=RANDOM_STATE).fit(X_train.loc[P_idx, FINAL])
Z = pca.transform(X_train.loc[P_idx, FINAL])
ev = pca.explained_variance_ratio_
fig, ax = plt.subplots(1, 2, figsize=(18, 7), gridspec_kw={"width_ratios": [1, 2]})
ax[0].bar(range(1, 11), 100 * ev, color="tab:blue"); ax[0].plot(range(1, 11), 100 * ev.cumsum(), "o-", color="tab:orange")
ax[0].set_xlabel("component"); ax[0].set_ylabel("% of variance"); ax[0].set_title("Explained variance (bars) and cumulative (line)")
fams = train.loc[P_idx, "attack_family"].to_numpy()
for f in FAMILY_ORDER:
    m = fams == f
    ax[1].scatter(Z[m, 0], Z[m, 1], s=4, alpha=.35, color=PAL[f], label=f)
ax[1].set_xlabel(f"PC1 ({100 * ev[0]:.1f} %)"); ax[1].set_ylabel(f"PC2 ({100 * ev[1]:.1f} %)")
ax[1].legend(markerscale=4, fontsize=8); ax[1].set_title("Training windows on the first two components (5,000 per family)")
plt.tight_layout(); savefig("pca")
load = pd.DataFrame(pca.components_[:2].T, index=FINAL, columns=["PC1", "PC2"])
print(f"PC1 + PC2 explain {100 * ev[:2].sum():.1f} % of the variance; 10 components: {100 * ev.sum():.1f} %")
top_load = {}
for pc in ["PC1", "PC2"]:
    t = load[pc].reindex(load[pc].abs().sort_values(ascending=False).index).head(6)
    top_load[f"{pc} feature"], top_load[f"{pc} loading"] = t.index.to_list(), t.round(3).to_list()
pd.DataFrame(top_load)

**Interpretation.** _(to be written: pca)_

---
## 11 · Summary

**Interpretation.** _(to be written: summary)_